In [ ]:
import pandas as pd
import os
import glob

# ==========================================
# 1. KONFIGURASI FOLDER
# ==========================================
INPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/RAW'  

# Mencari semua file berakhiran .json di dalam folder RAW
json_files = glob.glob(os.path.join(INPUT_DIR, '*.json'))

if not json_files:
    print("❌ Tidak ada file JSON yang ditemukan di folder RAW!")
    exit()

print(f"✅ Ditemukan {len(json_files)} file JSON. Memulai konversi...\n")

# ==========================================
# 2. PROSES KONVERSI JSON KE CSV
# ==========================================
for json_path in json_files:
    filename = os.path.basename(json_path)
    
    # Membuat nama file CSV (misal: Kuta_GR_RAW.json -> Kuta_GR_RAW.csv)
    csv_filename = filename.replace('.json', '.csv')
    csv_path = os.path.join(INPUT_DIR, csv_filename)
    
    try:
        print(f"⏳ Memproses: {filename}...")
        
        # Membaca file JSON
        # Apify biasanya menghasilkan list of dictionaries, jadi read_json standar sudah cukup
        df = pd.read_json(json_path)
        
        # Menyimpan ke format CSV
        df.to_csv(csv_path, index=False)
        
        print(f"   ✅ Berhasil! Tersimpan sebagai: {csv_filename} ({len(df):,} baris)")
        
    except Exception as e:
        print(f"   ❌ Gagal mengonversi {filename}. Error: {e}")

print("\n🚀 SEMUA FILE SELESAI DIKONVERSI!")
print("Sekarang kamu bisa menggunakan file .csv tersebut untuk tahap filtering kolom.")

✅ Ditemukan 6 file JSON. Memulai konversi...

⏳ Memproses: Seminyak_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: Seminyak_GR_RAW.csv (5,200 baris)
⏳ Memproses: Kuta_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: Kuta_GR_RAW.csv (10,000 baris)
⏳ Memproses: Pandawa_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: Pandawa_GR_RAW.csv (10,000 baris)
⏳ Memproses: Kelingking_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: Kelingking_GR_RAW.csv (10,000 baris)
⏳ Memproses: NusaDua_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: NusaDua_GR_RAW.csv (1,608 baris)
⏳ Memproses: PadangPadang_GR_RAW.json...
   ✅ Berhasil! Tersimpan sebagai: PadangPadang_GR_RAW.csv (9,149 baris)

🚀 SEMUA FILE SELESAI DIKONVERSI!
Sekarang kamu bisa menggunakan file .csv tersebut untuk tahap filtering kolom.


In [28]:
import pandas as pd
import os
import glob

# ==========================================
# 1. KONFIGURASI FILE & FOLDER
# ==========================================
INPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/RAW'  
OUTPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai'  
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengambil semua file yang berakhiran _GR_RAW.csv
file_pattern = os.path.join(INPUT_DIR, '*_GR_RAW.csv')
ta_files = glob.glob(file_pattern)

if not ta_files:
    print("❌ Tidak ada file _GR_RAW.csv yang ditemukan di direktori!")
    exit()

print(f"✅ Ditemukan {len(ta_files)} file TripAdvisor RAW.\n")

# Dictionary untuk menyimpan persentase null per file
null_percentages = {}
all_columns_set = set()

# ==========================================
# 2. BACA DATA & HITUNG MISSING VALUES
# ==========================================
for file_path in ta_files:
    # Mempercantik nama kolom header (misal: "Kuta" dari "Kuta_TA_RAW.csv")
    filename = os.path.basename(file_path)
    beach_name = filename.split('_')[0] 
    
    # Membaca seluruh baris untuk menghitung null
    df = pd.read_csv(file_path)
    
    # Update daftar total kolom unik dari semua file
    all_columns_set.update(df.columns)
    
    total_rows = len(df)
    # Menghitung persentase null
    null_pct = (df.isnull().sum() / total_rows * 100).round(2)
    
    # Menyimpan informasi persentase ke dalam dictionary berdasarkan nama pantai
    null_percentages[beach_name] = null_pct

# ==========================================
# 3. MEMBUAT TABEL GABUNGAN
# ==========================================
# Membuat DataFrame dari dictionary. Index-nya adalah nama kolom asli.
df_summary = pd.DataFrame(null_percentages)

# Jika ada kolom yang tidak ada di salah satu pantai, pandas akan mengisi NaN.
# Kita ganti NaN menjadi 100 (karena berarti 100% data di kolom itu hilang/tidak ada untuk pantai tersebut).
df_summary = df_summary.fillna(100.0)

# Menambahkan kolom "Rata-rata Null (%)" untuk mengurutkan tabel dari null tertinggi ke terendah secara global
df_summary['Avg_Null(%)'] = df_summary.mean(axis=1).round(2)
df_summary = df_summary.sort_values(by='Avg_Null(%)', ascending=False)

# Memformat output agar lebih rapi (menambahkan simbol % di tampilan print)
df_print = df_summary.copy()
for col in df_print.columns:
    df_print[col] = df_print[col].apply(lambda x: f"{x:.2f}%")

# ==========================================
# 4. TAMPILKAN HASIL TABEL
# ==========================================
print("="*100)
print("TABEL PERSENTASE MISSING VALUES (NULL) PER PANTAI")
print("="*100)

# Mengatur pandas agar saat diprint di terminal tidak terpotong ke bawah/menyamping
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df_print.to_string())
print("="*100)

✅ Ditemukan 7 file TripAdvisor RAW.

TABEL PERSENTASE MISSING VALUES (NULL) PER PANTAI
                         NusaDua  Pandawa     Kuta Kelingking    Sanur Seminyak PadangPadang Avg_Null(%)
visitedIn                100.00%  100.00%  100.00%    100.00%  100.00%  100.00%      100.00%     100.00%
rating                   100.00%  100.00%  100.00%    100.00%  100.00%  100.00%      100.00%     100.00%
responseFromOwnerDate    100.00%  100.00%  100.00%    100.00%  100.00%  100.00%      100.00%     100.00%
responseFromOwnerText    100.00%  100.00%  100.00%    100.00%  100.00%  100.00%      100.00%     100.00%
translatedLanguage        68.78%   54.91%   60.48%     69.18%   66.36%   74.63%       67.93%      66.04%
textTranslated            68.78%   54.91%   60.48%     69.18%   66.36%   74.63%       67.93%      66.04%
originalLanguage          45.46%   41.11%   41.32%     39.81%   45.38%   47.75%       38.97%      42.83%
text                      45.46%   41.11%   41.32%     39.81%   45.38%   

In [29]:
import pandas as pd
import os
import glob

# ==========================================
# 1. KONFIGURASI FOLDER & KOLOM TARGET
# ==========================================
INPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/RAW'
OUTPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Daftar kolom yang ingin dipertahankan khusus untuk Google Reviews (GR)
TARGET_COLUMNS_GR = [
    # --- METADATA DESTINASI ---
    'title',            # Di Google Maps, kolom 'title' berisi nama tempat wisata (misal: "Padang Padang Beach")
    
    # --- IDENTITAS REVIEW & URL ---
    'reviewId',         # ID unik ulasan (Krusial untuk deduplikasi)
    'reviewUrl',        # Link langsung ke ulasan
    
    # --- IDENTITAS PENGGUNA ---
    'reviewerId',       # ID Pengguna
    'name',             # Nama Pengguna
    
    # --- KONTEKS WAKTU ---
    'publishedAtDate',  # Tanggal dan waktu publikasi ulasan
    
    # --- ISI ULASAN (INTI NLP) ---
    'stars',            # Rating bintang yang diberikan (1-5)
    'text',             # Isi ulasan dalam bahasa asli
    'textTranslated',   # Isi ulasan terjemahan bahasa Inggris (bawaan Google)
]


# Mencari semua file _GR_RAW.csv
file_pattern = os.path.join(INPUT_DIR, '*_GR_RAW.csv')
gr_files = glob.glob(file_pattern)

if not gr_files:
    print("❌ Tidak ada file _GR_RAW.csv yang ditemukan di folder RAW!")
    exit()

print(f"✅ Ditemukan {len(gr_files)} file Google Reviews RAW.\n")

# ==========================================
# 2. VALIDASI KOLOM (CEK KETERSEDIAAN)
# ==========================================
print("="*70)
print("TAHAP 1: PENGECEKAN KOLOM TARGET DI SETIAP FILE")
print("="*70)

all_valid = True
for file_path in ta_files:
    filename = os.path.basename(file_path)
    # Baca header saja untuk validasi cepat
    df_temp = pd.read_csv(file_path, nrows=0)
    existing_cols = set(df_temp.columns)
    
    missing_cols = [col for col in TARGET_COLUMNS_GR if col not in existing_cols]
    
    if missing_cols:
        all_valid = False
        print(f"⚠️ {filename} -> KURANG KOLOM: {missing_cols}")
    else:
        print(f"✅ {filename} -> Lengkap (11 kolom tersedia)")

print("-" * 70)
if all_valid:
    print("🎉 SEMUA FILE MEMILIKI KOLOM YANG LENGKAP!")
else:
    print("⚠️ Ada file yang kehilangan beberapa kolom target.")
    print("   Sistem akan otomatis menambahkan kolom tersebut dengan isi kosong (NaN) agar format seragam.")

# ==========================================
# 3. EKSTRAKSI DAN PENYIMPANAN KE FILE BARU
# ==========================================
print("\n" + "="*70)
print(f"TAHAP 2: EKSTRAKSI & SIMPAN KE FOLDER {OUTPUT_DIR}")
print("="*70)

for file_path in ta_files:
    filename = os.path.basename(file_path)
    # Hapus '_RAW' dari nama file (misal: Kuta_TA_RAW.csv -> Kuta_TA.csv)
    new_filename = filename.replace('_RAW.csv', '.csv')
    output_path = os.path.join(OUTPUT_DIR, new_filename)
    
    # Baca seluruh data
    df = pd.read_csv(file_path)
    
    # Jika ada kolom target yang tidak ada di dataframe, buat kolomnya dan isi dengan NaN
    for col in TARGET_COLUMNS_GR:
        if col not in df.columns:
            df[col] = pd.NA
            
    # Filter hanya kolom target dan urutkan sesuai list TARGET_COLUMNS_GR
    df_filtered = df[TARGET_COLUMNS_GR].copy()
    
    # Simpan ke folder PerPantai
    df_filtered.to_csv(output_path, index=False)
    
    print(f"💾 Tersimpan: {new_filename} ({len(df_filtered)} baris, {len(df_filtered.columns)} kolom)")

print("\n🚀 PROSES SELESAI! Seluruh file siap digunakan untuk tahap Preprocessing NLP.")

✅ Ditemukan 7 file Google Reviews RAW.

TAHAP 1: PENGECEKAN KOLOM TARGET DI SETIAP FILE
✅ NusaDua_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Pandawa_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Kuta_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Kelingking_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Sanur_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Seminyak_GR_RAW.csv -> Lengkap (11 kolom tersedia)
✅ PadangPadang_GR_RAW.csv -> Lengkap (11 kolom tersedia)
----------------------------------------------------------------------
🎉 SEMUA FILE MEMILIKI KOLOM YANG LENGKAP!

TAHAP 2: EKSTRAKSI & SIMPAN KE FOLDER /Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai
💾 Tersimpan: NusaDua_GR.csv (1608 baris, 9 kolom)
💾 Tersimpan: Pandawa_GR.csv (10000 baris, 9 kolom)
💾 Tersimpan: Kuta_GR.csv (10000 baris, 9 kolom)
💾 Tersimpan: Kelingking_GR.csv (10000 baris, 9 kolom)
💾 Tersimpan: Sanur_GR.csv (5000 baris, 9 kolom)
💾 Tersimpan: Seminyak_GR.csv (5200 baris, 9 kolom)
💾 Tersimpan: Pa